In [1]:
from pathlib import Path
import duckdb

cwd = Path.cwd()
root = cwd.parent if cwd.name == 'notebooks' else cwd
db_path = root / "data" / "fdic.duckdb"
assert db_path.exists(), f"[duckdb_err]: path {db_path} not found"

with duckdb.connect(str(db_path), read_only=True) as conn:
    df = conn.sql("SELECT * from fct_bank_quarter_training").df()
    print(df.shape) 
    print(df.head().T)


(681297, 20)
                                                             0  \
cert                                                     32266   
report_date                                2001-03-31 00:00:00   
bank_quarter_id                                 32266_20010331   
failure_in_year                                              0   
total_risk_based_capital_pct                         12.008618   
nonperforming_assets_to_assets_pct                    0.183577   
loan_loss_allowance_to_loans_pct                        0.5272   
roa_pct                                                 0.4986   
net_interest_margin_pct                               1.919209   
efficiency_ratio_pct                                 62.743191   
brokered_deposits_pct                                61.592887   
core_deposits_pct                                    48.645647   
fhlb_advances_to_assets_pct                          21.563464   
construction_concentration_pct                       13.342318 

In [2]:
import pandas as pd

df['report_date'] = pd.to_datetime(df['report_date'])

train_split_dt = pd.Timestamp('2008-12-31')
test_split_dt = pd.Timestamp('2010-01-01')

train_df = df[df['report_date'] <= train_split_dt].copy()
test_df = df[df['report_date'] >= test_split_dt].copy()

print(f"Train dataframe: {train_df.shape}; Test dataframe: {test_df.shape}")

print(df['failure_in_year'].sum())
print(train_df['failure_in_year'].sum())
print(test_df['failure_in_year'].sum())


Train dataframe: (286499, 20); Test dataframe: (362408, 20)
2305
530
1109


In [3]:
feature_columns = [
    'nonperforming_assets_to_assets_pct', 
    'loan_loss_allowance_to_loans_pct', 
    'roa_pct', 
    'net_interest_margin_pct', 
    'efficiency_ratio_pct', 
    'brokered_deposits_pct', 
    'core_deposits_pct', 
    'fhlb_advances_to_assets_pct', 
    'construction_concentration_pct', 
    'leverage_ratio_pct', 
    'core_deposits_pct_4q_chg', 
    'nonperforming_assets_to_assets_pct_4q_chg', 
    'roa_pct_4q_chg', 
    'leverage_ratio_pct_4q_chg'
]

x_train = train_df[feature_columns]
y_train = train_df['failure_in_year']

x_test = test_df[feature_columns]
y_test = test_df['failure_in_year']


x_train_null_cnt = x_train.isnull().sum()
print(x_train_null_cnt)


nonperforming_assets_to_assets_pct               0
loan_loss_allowance_to_loans_pct                 0
roa_pct                                          0
net_interest_margin_pct                          0
efficiency_ratio_pct                             0
brokered_deposits_pct                          682
core_deposits_pct                                0
fhlb_advances_to_assets_pct                    601
construction_concentration_pct                 567
leverage_ratio_pct                             602
core_deposits_pct_4q_chg                     42516
nonperforming_assets_to_assets_pct_4q_chg    42516
roa_pct_4q_chg                               42516
leverage_ratio_pct_4q_chg                    43031
dtype: int64


In [4]:
x_train["is_first_year"] = (x_train["core_deposits_pct_4q_chg"].isna() & 
                            x_train["nonperforming_assets_to_assets_pct_4q_chg"].isna() & 
                            x_train["roa_pct_4q_chg"].isna() & 
                            x_train["leverage_ratio_pct_4q_chg"].isna()
                            ).astype(int)

x_test["is_first_year"] = (x_test["core_deposits_pct_4q_chg"].isna() & 
                            x_test["nonperforming_assets_to_assets_pct_4q_chg"].isna() & 
                            x_test["roa_pct_4q_chg"].isna() & 
                            x_test["leverage_ratio_pct_4q_chg"].isna()
                            ).astype(int)

In [5]:
x_train['nonperforming_assets_to_assets_pct'] = x_train['nonperforming_assets_to_assets_pct'].copy().fillna(x_train['nonperforming_assets_to_assets_pct'].median())
x_train['loan_loss_allowance_to_loans_pct'] = x_train['loan_loss_allowance_to_loans_pct'].copy().fillna(x_train['loan_loss_allowance_to_loans_pct'].median())
x_train['roa_pct'] = x_train['roa_pct'].copy().fillna(x_train['roa_pct'].median())
x_train['net_interest_margin_pct'] = x_train['net_interest_margin_pct'] .copy().fillna(x_train['net_interest_margin_pct'] .median())
x_train['efficiency_ratio_pct'] = x_train['efficiency_ratio_pct'].copy().fillna(x_train['efficiency_ratio_pct'].median())
x_train['brokered_deposits_pct'] = x_train['brokered_deposits_pct'].copy().fillna(x_train['brokered_deposits_pct'].median())
x_train['core_deposits_pct'] = x_train['core_deposits_pct'].copy().fillna(x_train['core_deposits_pct'].median())
x_train['fhlb_advances_to_assets_pct'] = x_train['fhlb_advances_to_assets_pct'].copy().fillna(x_train['fhlb_advances_to_assets_pct'].median())
x_train['construction_concentration_pct'] = x_train['construction_concentration_pct'].copy().fillna(x_train['construction_concentration_pct'].median())
x_train['leverage_ratio_pct'] = x_train['leverage_ratio_pct'].copy().fillna(x_train['leverage_ratio_pct'].median())
x_train['core_deposits_pct_4q_chg'] = x_train['core_deposits_pct_4q_chg'].copy().fillna(x_train['core_deposits_pct_4q_chg'].median())
x_train['nonperforming_assets_to_assets_pct_4q_chg'] = x_train['nonperforming_assets_to_assets_pct_4q_chg'].copy().fillna(x_train['nonperforming_assets_to_assets_pct_4q_chg'].median())
x_train['roa_pct_4q_chg'] = x_train['roa_pct_4q_chg'].copy().fillna(x_train['roa_pct_4q_chg'].median())
x_train['leverage_ratio_pct_4q_chg'] = x_train['leverage_ratio_pct_4q_chg'].copy().fillna(x_train['leverage_ratio_pct_4q_chg'].median())

In [6]:
x_test['nonperforming_assets_to_assets_pct']        = x_test['nonperforming_assets_to_assets_pct'].fillna(x_train['nonperforming_assets_to_assets_pct'].median())
x_test['loan_loss_allowance_to_loans_pct']          = x_test['loan_loss_allowance_to_loans_pct'].fillna(x_train['loan_loss_allowance_to_loans_pct'].median())
x_test['roa_pct']                                   = x_test['roa_pct'].fillna(x_train['roa_pct'].median())
x_test['net_interest_margin_pct']                   = x_test['net_interest_margin_pct'] .fillna(x_train['net_interest_margin_pct'] .median())
x_test['efficiency_ratio_pct']                      = x_test['efficiency_ratio_pct'].fillna(x_train['efficiency_ratio_pct'].median())
x_test['brokered_deposits_pct']                     = x_test['brokered_deposits_pct'].fillna(x_train['brokered_deposits_pct'].median())
x_test['core_deposits_pct']                         = x_test['core_deposits_pct'].fillna(x_train['core_deposits_pct'].median())
x_test['fhlb_advances_to_assets_pct']               = x_test['fhlb_advances_to_assets_pct'].fillna(x_train['fhlb_advances_to_assets_pct'].median())
x_test['construction_concentration_pct']            = x_test['construction_concentration_pct'].fillna(x_train['construction_concentration_pct'].median())
x_test['leverage_ratio_pct']                        = x_test['leverage_ratio_pct'].fillna(x_train['leverage_ratio_pct'].median())
x_test['core_deposits_pct_4q_chg']                  = x_test['core_deposits_pct_4q_chg'].fillna(x_train['core_deposits_pct_4q_chg'].median())
x_test['nonperforming_assets_to_assets_pct_4q_chg'] = x_test['nonperforming_assets_to_assets_pct_4q_chg'].fillna(x_train['nonperforming_assets_to_assets_pct_4q_chg'].median())
x_test['roa_pct_4q_chg']                            = x_test['roa_pct_4q_chg'].fillna(x_train['roa_pct_4q_chg'].median())
x_test['leverage_ratio_pct_4q_chg']                 = x_test['leverage_ratio_pct_4q_chg'].fillna(x_train['leverage_ratio_pct_4q_chg'].median())

In [7]:
print(f"x-train nulls: \n{x_train.isnull().sum()}\n")


x-train nulls: 
nonperforming_assets_to_assets_pct           0
loan_loss_allowance_to_loans_pct             0
roa_pct                                      0
net_interest_margin_pct                      0
efficiency_ratio_pct                         0
brokered_deposits_pct                        0
core_deposits_pct                            0
fhlb_advances_to_assets_pct                  0
construction_concentration_pct               0
leverage_ratio_pct                           0
core_deposits_pct_4q_chg                     0
nonperforming_assets_to_assets_pct_4q_chg    0
roa_pct_4q_chg                               0
leverage_ratio_pct_4q_chg                    0
is_first_year                                0
dtype: int64



In [8]:
print(f"x-test nulls: \n{x_test.isnull().sum()}\n")


x-test nulls: 
nonperforming_assets_to_assets_pct           0
loan_loss_allowance_to_loans_pct             0
roa_pct                                      0
net_interest_margin_pct                      0
efficiency_ratio_pct                         0
brokered_deposits_pct                        0
core_deposits_pct                            0
fhlb_advances_to_assets_pct                  0
construction_concentration_pct               0
leverage_ratio_pct                           0
core_deposits_pct_4q_chg                     0
nonperforming_assets_to_assets_pct_4q_chg    0
roa_pct_4q_chg                               0
leverage_ratio_pct_4q_chg                    0
is_first_year                                0
dtype: int64



In [9]:
print(f"medians: \n{x_train.median(numeric_only=True)}")

medians: 
nonperforming_assets_to_assets_pct            0.459944
loan_loss_allowance_to_loans_pct              1.212504
roa_pct                                       0.993106
net_interest_margin_pct                       4.008031
efficiency_ratio_pct                         65.853659
brokered_deposits_pct                         0.000000
core_deposits_pct                            69.014564
fhlb_advances_to_assets_pct                   1.545738
construction_concentration_pct                4.477114
leverage_ratio_pct                            9.404728
core_deposits_pct_4q_chg                     -0.552769
nonperforming_assets_to_assets_pct_4q_chg     0.000000
roa_pct_4q_chg                               -0.008681
leverage_ratio_pct_4q_chg                     0.030701
is_first_year                                 0.000000
dtype: float64


In [10]:
prc_lims = x_train[feature_columns].quantile([0.001, 0.999])
lower, upper = prc_lims.loc[0.001], prc_lims.loc[0.999]

x_train[feature_columns] = x_train[feature_columns].clip(lower=lower, upper=upper, axis=1)
x_test[feature_columns]  = x_test[feature_columns].clip(lower=lower, upper=upper, axis=1)

In [11]:
mean = x_train[feature_columns].mean()
std = x_train[feature_columns].std()

x_train[feature_columns] = (x_train[feature_columns] - mean) / std
x_test[feature_columns] = (x_test[feature_columns] - mean) / std



In [12]:
print(x_train[feature_columns].agg(['mean', 'std']).round(2).T)


                                           mean  std
nonperforming_assets_to_assets_pct          0.0  1.0
loan_loss_allowance_to_loans_pct            0.0  1.0
roa_pct                                    -0.0  1.0
net_interest_margin_pct                    -0.0  1.0
efficiency_ratio_pct                        0.0  1.0
brokered_deposits_pct                       0.0  1.0
core_deposits_pct                          -0.0  1.0
fhlb_advances_to_assets_pct                 0.0  1.0
construction_concentration_pct             -0.0  1.0
leverage_ratio_pct                          0.0  1.0
core_deposits_pct_4q_chg                    0.0  1.0
nonperforming_assets_to_assets_pct_4q_chg   0.0  1.0
roa_pct_4q_chg                             -0.0  1.0
leverage_ratio_pct_4q_chg                  -0.0  1.0


In [13]:
print(x_test[feature_columns].agg(['mean', 'std']).round(2).T)

                                           mean   std
nonperforming_assets_to_assets_pct         0.43  1.74
loan_loss_allowance_to_loans_pct           0.17  1.07
roa_pct                                   -0.02  0.81
net_interest_margin_pct                   -0.33  0.89
efficiency_ratio_pct                      -0.02  0.55
brokered_deposits_pct                      0.02  0.90
core_deposits_pct                          0.75  0.83
fhlb_advances_to_assets_pct               -0.24  0.70
construction_concentration_pct            -0.22  0.58
leverage_ratio_pct                        -0.00  0.61
core_deposits_pct_4q_chg                   0.24  1.08
nonperforming_assets_to_assets_pct_4q_chg -0.19  1.21
roa_pct_4q_chg                             0.02  0.98
leverage_ratio_pct_4q_chg                  0.09  0.43


In [14]:
test_df['leverage_ratio_alarm'] = (test_df['leverage_ratio_pct'] < 5).astype(int)

In [15]:
caught = ((test_df['leverage_ratio_alarm'] == 1) & (y_test == 1)).sum() 
false_alarm = ((test_df['leverage_ratio_alarm'] == 1) & (y_test == 0)).sum()
missed = ((test_df['leverage_ratio_alarm'] == 0) & (y_test == 1)).sum()
corr_let_through = ((test_df['leverage_ratio_alarm'] == 0) & (y_test == 0)).sum()

print(f"caught: {caught}")
print(f"false alarm: {false_alarm}")
print(f"missed: {missed}")
print(f"correctly let through: {corr_let_through}")

recall = caught / (caught + missed)
precision = caught / (caught + false_alarm)

print(f"recall: {recall}")
print(f"precision: {precision}")

lev_limit = lower['leverage_ratio_pct']
clipped = train_df['leverage_ratio_pct'] < lev_limit

print("lower limit:", lev_limit)
print("rows clipped:", clipped.sum())
print("positives clipped:", (clipped & (train_df['failure_in_year'] == 1)).sum(), "of 530")

caught: 981
false alarm: 2725
missed: 128
correctly let through: 358574
recall: 0.8845807033363391
precision: 0.2647058823529412
lower limit: 4.160022050881054
rows clipped: 287
positives clipped: 85 of 530


In [16]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(x_train, y_train)

prob = model.predict_proba(x_test)[:, 1]

risks = pd.Series(prob, index=x_test.index)

top_rows_idx = risks.nlargest(3706).index

model_alarm = risks.index.isin(top_rows_idx).astype(int)
model_alarm = pd.Series(model_alarm, index=x_test.index)

print(model_alarm.sum())


caught = ((model_alarm == 1) & (y_test == 1)).sum()
missed = ((model_alarm == 0) & (y_test == 1)).sum()
false_alarm = ((model_alarm == 1) & (y_test == 0)).sum()

model_recall = caught / (caught + missed)
model_precision = caught / (caught + false_alarm)

print(f"model caught: {caught}")
print(f"model missed: {missed}")
print(f"model's false alarms: {false_alarm}")
print(f"model's precision: {model_precision}")
print(f"model's recall: {model_recall}")

3706
model caught: 874
model missed: 235
model's false alarms: 2832
model's precision: 0.2358337830545062
model's recall: 0.78809738503156


In [22]:
for f, w in zip(model.feature_names_in_, model.coef_[0]):
    print(f"{f} - {w}")

nonperforming_assets_to_assets_pct - 0.3754562636932734
loan_loss_allowance_to_loans_pct - -0.004658334451962886
roa_pct - -0.29246489825146355
net_interest_margin_pct - 0.10577692940269381
efficiency_ratio_pct - 0.0228492364607375
brokered_deposits_pct - 0.19264101226215527
core_deposits_pct - -0.4671311043639947
fhlb_advances_to_assets_pct - 0.0902450442801879
construction_concentration_pct - 0.2647053213960605
leverage_ratio_pct - -2.6223395675327015
core_deposits_pct_4q_chg - 0.15929745097963452
nonperforming_assets_to_assets_pct_4q_chg - 0.022303303994412958
roa_pct_4q_chg - -0.09001923227569598
leverage_ratio_pct_4q_chg - -0.03369199569301498
is_first_year - -0.15449876497489817
